# Notebook 01 — Train Money Classifier từ Roboflow VND Dataset

**Mục tiêu:** train `MobileNetV3-Small` phân loại 10 class (9 mệnh giá VND + 1 unknown), export `vnd_classifier.tflite` ~5MB.

**Dataset:** [Vietnamese Currency Detector (Roboflow)](https://universe.roboflow.com/cv-aal82/vietnamese-currency-detector) — 2569 ảnh, **multilabel-classification format** (folder per class, không phải object detection).

**Pipeline:**
1. Download Roboflow dataset (`folder` format) qua API key.
2. Re-organize folder theo class label chuẩn của project Android.
3. Bổ sung class `unknown` từ CIFAR-10 (negative samples).
4. Train MobileNetV3-Small với augmentation (RandomResizedCrop, ColorJitter, Rotation, GaussianBlur).
5. Export PyTorch → ONNX → TFLite INT8.
6. Verify class order khớp `MONEY_LABELS` trong `MoneyClassifier.kt`.

**Yêu cầu:**
- Google Colab runtime: **GPU T4** (free).
- Roboflow account → API key tại https://app.roboflow.com/settings/api
- Thời gian: ~30-60 phút trên T4.

**Output:**
- `/content/vnd_classifier.tflite` (~5 MB)
- `/content/vnd_labels.txt` (10 dòng)
- Copy cả 2 file vào `app/src/main/assets/ml/` trong project Android.

## 1. Setup & deps

In [ ]:
!pip install -q torch torchvision roboflow Pillow tqdm onnx tensorflow

In [ ]:
import os, shutil, random
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from PIL import Image
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## 2. Download dataset từ Roboflow

Paste API key của bạn vào cell dưới. Lấy tại https://app.roboflow.com/settings/api

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = 'YOUR_API_KEY_HERE'  # ← paste vào đây

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('cv-aal82').project('vietnamese-currency-detector')
# Project type: multilabel-classification → format hợp lệ: 'folder' (1 folder/class)
# Cấu trúc: {location}/train/<class_name>/*.jpg, /valid/..., /test/...
dataset = project.version(1).download('folder')
DATA_ROOT = Path(dataset.location)
print('Downloaded to:', DATA_ROOT)
print('Subdirs:', [p.name for p in DATA_ROOT.iterdir() if p.is_dir()])

In [ ]:
# Folder format không có data.yaml — class names chính là tên thư mục con trong train/
TRAIN_DIR = DATA_ROOT / 'train'
if not TRAIN_DIR.exists():
    # 1 số version Roboflow đặt là 'Train' viết hoa
    candidates = [p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.lower() == 'train']
    if candidates:
        TRAIN_DIR = candidates[0]

ROBOFLOW_CLASSES = sorted([p.name for p in TRAIN_DIR.iterdir() if p.is_dir()])
print('Roboflow class folders:', ROBOFLOW_CLASSES)
print('Number of classes:', len(ROBOFLOW_CLASSES))
for c in ROBOFLOW_CLASSES:
    n = len(list((TRAIN_DIR / c).glob('*.*')))
    print(f'  {c}: {n} images')

## 3. Re-organize folder theo class label chuẩn

Roboflow đã có cấu trúc folder per class — chỉ cần map tên class của Roboflow → label chuẩn của Android (`500000`, `200000`, ..., `unknown`).

**Verify mapping bằng tay** sau cell tiếp theo: nếu Roboflow đặt tên class lạ (vd `1k_VND`, `5_000`, `bill_500k`), update hàm `roboflow_name_to_vnd`.

In [ ]:
# Map từ Roboflow class name → mệnh giá VND (theo MONEY_LABELS trong Kotlin)
# Edit dictionary nếu tên class khác. Print ROBOFLOW_CLASSES ở cell trên để xem chính xác.
import re

def roboflow_name_to_vnd(name: str) -> int:
    """Heuristic: trích số từ tên class. '1000' → 1000, '1k' → 1000, '500k_VND' → 500000."""
    s = name.lower().replace(',', '').replace('.', '').replace('_', '').replace(' ', '')
    # Pattern: optional digits + optional 'k' suffix + optional 'vnd'/'đ'
    m = re.match(r'(\d+)(k)?', s)
    if not m:
        return -1
    n = int(m.group(1))
    if m.group(2) == 'k':
        n *= 1000
    return n

ROBOFLOW_NAME_TO_VND = {name: roboflow_name_to_vnd(name) for name in ROBOFLOW_CLASSES}
print('Mapping (verify bằng mắt):')
for k, v in ROBOFLOW_NAME_TO_VND.items():
    print(f'  {k!r:30s} → {v:>8} VND')

In [ ]:
# Class order PHẢI khớp MONEY_LABELS trong MoneyClassifier.kt
# (index 0 = 500k, ..., index 8 = 1k, index 9 = unknown)
OUTPUT_CLASSES = [500_000, 200_000, 100_000, 50_000, 20_000, 10_000, 5_000, 2_000, 1_000, 0]
OUTPUT_LABELS  = ['500000', '200000', '100000', '50000', '20000', '10000', '5000', '2000', '1000', 'unknown']

VND_TO_INDEX = {vnd: i for i, vnd in enumerate(OUTPUT_CLASSES)}
print('Class index mapping (Android expects):')
for vnd, idx in VND_TO_INDEX.items():
    print(f'  index {idx} → {vnd} VND ({OUTPUT_LABELS[idx]})')

In [ ]:
# Copy ảnh từ Roboflow folder → /content/classification/<split>/<label>/img.jpg
# Map Roboflow class folder → Android label (qua VND amount)
OUTPUT_ROOT = Path('/content/classification')
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

copied = 0
skipped = 0
for split in ['train', 'valid', 'test']:
    split_dir = DATA_ROOT / split
    if not split_dir.exists():
        # fallback case sensitivity
        candidates = [p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.lower() == split]
        if not candidates:
            continue
        split_dir = candidates[0]

    target_split = 'val' if split == 'valid' else split

    for class_dir in split_dir.iterdir():
        if not class_dir.is_dir():
            continue
        cls_name = class_dir.name
        vnd = ROBOFLOW_NAME_TO_VND.get(cls_name, -1)
        if vnd not in VND_TO_INDEX:
            # Mệnh giá không nằm trong 9 mệnh giá → gán unknown
            out_label = 'unknown'
        else:
            out_label = OUTPUT_LABELS[VND_TO_INDEX[vnd]]

        out_dir = OUTPUT_ROOT / target_split / out_label
        out_dir.mkdir(parents=True, exist_ok=True)

        for img_path in class_dir.iterdir():
            if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
                skipped += 1
                continue
            # Append class name vào filename để tránh collision khi nhiều class map về cùng label
            dst = out_dir / f'{cls_name}_{img_path.name}'
            shutil.copy(img_path, dst)
            copied += 1

print(f'\nCopied {copied} images, skipped {skipped} non-image files\n')
for split in ['train', 'val', 'test']:
    split_dir = OUTPUT_ROOT / split
    if not split_dir.exists():
        continue
    print(f'{split}:')
    for cls in sorted(split_dir.iterdir()):
        print(f'  {cls.name:>10}: {len(list(cls.iterdir()))}')

## 4. Bổ sung class `unknown` (negative samples)

Roboflow dataset chỉ có ảnh tiền — model sẽ overfit, không biết nói "không phải tiền". Cần thêm ảnh non-money. Cách rẻ nhất: sample từ ImageNet hoặc CIFAR-100 (random objects).

In [ ]:
# Download CIFAR-10 (60k ảnh objects ngẫu nhiên), lấy ~200 ảnh làm 'unknown'
from torchvision.datasets import CIFAR10
cifar = CIFAR10(root='/content/cifar', train=True, download=True)

unknown_train = OUTPUT_ROOT / 'train' / 'unknown'
unknown_val = OUTPUT_ROOT / 'val' / 'unknown'
unknown_train.mkdir(parents=True, exist_ok=True)
unknown_val.mkdir(parents=True, exist_ok=True)

random.seed(42)
indices = random.sample(range(len(cifar)), 250)
for i, idx in enumerate(indices):
    img, _ = cifar[idx]
    target = unknown_val if i < 50 else unknown_train
    img.resize((224, 224)).save(target / f'cifar_{i:04d}.jpg', quality=85)

print('Unknown class samples added (train + val)')

## 5. Dataset + augmentation

In [ ]:
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.GaussianBlur(kernel_size=3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# QUAN TRỌNG: ép thứ tự class theo OUTPUT_LABELS để model output đúng index Android expect
class FixedOrderImageFolder(datasets.ImageFolder):
    def find_classes(self, directory):
        # Trả về (classes_in_fixed_order, class_to_idx)
        present = {d.name for d in Path(directory).iterdir() if d.is_dir()}
        ordered = [c for c in OUTPUT_LABELS if c in present]
        class_to_idx = {c: OUTPUT_LABELS.index(c) for c in ordered}
        return ordered, class_to_idx

train_ds = FixedOrderImageFolder(OUTPUT_ROOT / 'train', transform=train_tf)
val_ds = FixedOrderImageFolder(OUTPUT_ROOT / 'val', transform=val_tf)
print('Train:', len(train_ds), 'Val:', len(val_ds))
print('class_to_idx:', train_ds.class_to_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

## 6. Model: MobileNetV3-Small fine-tune

In [ ]:
NUM_CLASSES = 10  # 9 mệnh giá + unknown

model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
# Output layer: 1024 → NUM_CLASSES
model.classifier[3] = nn.Linear(model.classifier[3].in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

print(model.classifier)

In [ ]:
EPOCHS = 30
best_val_acc = 0.0
best_state = None

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} train', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
        train_correct += (out.argmax(1) == labels).sum().item()
        train_total += imgs.size(0)
    train_acc = train_correct / train_total

    # Val
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            val_correct += (out.argmax(1) == labels).sum().item()
            val_total += imgs.size(0)
    val_acc = val_correct / val_total

    scheduler.step()
    print(f'Epoch {epoch+1:02d}/{EPOCHS} | loss {train_loss/train_total:.3f} | train_acc {train_acc:.3f} | val_acc {val_acc:.3f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print(f'\nBest val_acc: {best_val_acc:.3f}')
model.load_state_dict(best_state)
torch.save(model.state_dict(), '/content/best_model.pt')

## 7. Confusion matrix trên test set

In [ ]:
import numpy as np
test_dir = OUTPUT_ROOT / 'test'
if test_dir.exists():
    test_ds = FixedOrderImageFolder(test_dir, transform=val_tf)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    model.eval()
    with torch.no_grad():
        for imgs, labels in test_loader:
            preds = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
            for p, l in zip(preds, labels.numpy()):
                cm[l][p] += 1
    print('Confusion matrix (rows=truth, cols=pred):')
    print('        ', '  '.join(f'{c:>7}' for c in OUTPUT_LABELS))
    for i, row in enumerate(cm):
        print(f'{OUTPUT_LABELS[i]:>8}', '  '.join(f'{v:>7}' for v in row))
    overall_acc = cm.trace() / cm.sum()
    print(f'\nTest accuracy: {overall_acc:.3f}')
else:
    print('No test split — skipping confusion matrix')

## 8. Export PyTorch → ONNX → TFLite INT8

In [ ]:
# 8.1 PyTorch → ONNX
model.eval().cpu()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(
    model, dummy, '/content/vnd_classifier.onnx',
    input_names=['input'], output_names=['logits'],
    opset_version=13,
)
print('ONNX saved')

In [ ]:
# 8.2 ONNX → TFLite qua onnx2tf
# onnx-tf bị break voi ONNX 1.16+ (`from onnx import mapping` đã bị remove).
# onnx2tf là replacement actively maintained, output trực tiếp TFLite.
!pip install -q onnx2tf onnxruntime tf-keras sng4onnx onnx-graphsurgeon

import subprocess
ret = subprocess.run([
    'onnx2tf',
    '-i', '/content/vnd_classifier.onnx',
    '-o', '/content/vnd_classifier_tf',
], capture_output=True, text=True)
print(ret.stdout[-3000:])
if ret.returncode != 0:
    print('STDERR:', ret.stderr[-2000:])

import os
print('\nGenerated files:')
for f in sorted(os.listdir('/content/vnd_classifier_tf')):
    p = f'/content/vnd_classifier_tf/{f}'
    if os.path.isfile(p):
        print(f'  {f}: {os.path.getsize(p) // 1024} KB')

In [ ]:
# 8.3 Pick file TFLite tu output cua onnx2tf
# onnx2tf tu dong tao ra file *_float32.tflite (FP32). Co the bo sung --output_integer_quantized_tflite
# de them ban INT8, nhung FP32 ~5-6 MB la chap nhan duoc cho APK.
import shutil, os, glob

tflite_files = glob.glob('/content/vnd_classifier_tf/*.tflite')
print('TFLite files generated:', [os.path.basename(f) for f in tflite_files])

# Uu tien file float32 (chinh xac cao nhat). Neu khong co thi lay file dau tien.
fp32_files = [f for f in tflite_files if 'float32' in f.lower()]
src = fp32_files[0] if fp32_files else (tflite_files[0] if tflite_files else None)
assert src is not None, 'Khong tim thay file .tflite — onnx2tf da fail. Xem log cell tren.'

shutil.copy(src, '/content/vnd_classifier.tflite')
size_kb = os.path.getsize('/content/vnd_classifier.tflite') // 1024
print(f'\nFinal: /content/vnd_classifier.tflite ({size_kb} KB)')

# Optional: tao them ban dynamic-range quantized (smaller, INT8 weights + FP activations) qua TF API
# de so sanh size. Bo qua neu chi can FP32.
print('\nTip: Neu muon INT8 (size ~1.5MB), chay lai onnx2tf voi flag:')
print("  onnx2tf -i model.onnx -o out_dir -oiqd  # output INT8 quantized")

In [ ]:
# 8.4 Verify TFLite chay duoc + sanity check class order
import tensorflow as tf
import numpy as np

interpreter = tf.lite.Interpreter(model_path='/content/vnd_classifier.tflite')
interpreter.allocate_tensors()
in_details = interpreter.get_input_details()[0]
out_details = interpreter.get_output_details()[0]
print('Input  shape:', in_details['shape'], 'dtype:', in_details['dtype'])
print('Output shape:', out_details['shape'], 'dtype:', out_details['dtype'])

# Chay 1 anh tu val_ds → in top-3 prediction
img, true_lbl = val_ds[0]
# PyTorch [C, H, W] → TFLite [1, H, W, C]
inp = img.unsqueeze(0).permute(0, 2, 3, 1).numpy().astype(np.float32)
# onnx2tf co the output input shape NCHW hoac NHWC tuy version, check va transpose neu can
if list(in_details['shape']) == list(inp.shape[:1]) + [inp.shape[3], inp.shape[1], inp.shape[2]]:
    inp = inp.transpose(0, 3, 1, 2)  # back to NCHW

interpreter.set_tensor(in_details['index'], inp)
interpreter.invoke()
out = interpreter.get_tensor(out_details['index'])[0]

print(f'\nTruth: {OUTPUT_LABELS[true_lbl]}')
top3 = np.argsort(out)[::-1][:3]
for i in top3:
    print(f'  pred {OUTPUT_LABELS[i]}: logit={out[i]:.3f}')

## 9. Save labels file + zip output

In [ ]:
with open('/content/vnd_labels.txt', 'w') as f:
    f.write('\n'.join(OUTPUT_LABELS) + '\n')

import os
for fname in ['vnd_classifier.tflite', 'vnd_labels.txt']:
    path = f'/content/{fname}'
    size_kb = os.path.getsize(path) // 1024
    print(f'  {fname}: {size_kb} KB')

print('\n=== NEXT STEPS ===')
print('1. Download 2 files (Files panel → right-click → Download):')
print('     /content/vnd_classifier.tflite')
print('     /content/vnd_labels.txt')
print('2. Copy vào D:/hotronguoikhiemthi/app/src/main/assets/ml/')
print('3. Rebuild: .\\gradlew.bat installDebug')
print('4. TfliteMoneyClassifier sẽ tự load file thay vì fallback FakeMoneyClassifier.')